## 고객 특성별 정상 전환 검정

### 분석 목적
- `gender`, `age_group`, `income_group`이 `is_normal_flow`와 관련이 있는지 확인
- 즉, 고객 특성이 **정상 전환 여부**를 설명하는지 살펴봄

### 분석 범위
- `bogo`와 `discount`만 사용
- `informational`은 `is_normal_flow`가 항상 0이어서 정상 전환 분석에서 제외

### 가설
- H0: 고객 특성과 정상 전환 여부는 독립이다
- H1: 고객 특성과 정상 전환 여부는 독립이 아니다


In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path

from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from IPython.display import display


In [2]:
# 데이터 불러오기
candidate_paths = [
    Path('data/promotion_df_final.csv'),
    Path('../data/promotion_df_final.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('promotion_df_final.csv not found in data/ or ../data/')

promotion_final = pd.read_csv(data_path)

# 정상 전환 분석은 bogo / discount 범위에서만 진행
analysis_df = promotion_final[promotion_final['offer_type'].isin(['bogo', 'discount'])].copy()

# 고객 특성 정리
analysis_df['gender'] = analysis_df['gender'].fillna('Unknown')

analysis_df['age_group'] = pd.cut(
    analysis_df['age'],
    bins=[0, 19, 29, 39, 49, 59, 69, 120],
    labels=['10s', '20s', '30s', '40s', '50s', '60s', '70+']
).astype('object').fillna('Unknown')

analysis_df['income_group'] = pd.cut(
    analysis_df['income'],
    bins=[0, 40000, 60000, 80000, 100000, np.inf],
    labels=['0-40k', '40-60k', '60-80k', '80-100k', '100k+'],
    right=False
).astype('object').fillna('Unknown')

analysis_df['is_normal_flow'] = analysis_df['is_normal_flow'].fillna(0).astype(int)

display(analysis_df[['offer_type', 'gender', 'age_group', 'income_group', 'is_normal_flow']].head())
print('shape:', analysis_df.shape)
print(analysis_df['offer_type'].value_counts())


,offer_type,gender,age_group,income_group,is_normal_flow
0,bogo,M,30s,60-80k,0
1,discount,M,30s,60-80k,0
2,discount,M,30s,60-80k,0
3,discount,O,40s,40-60k,1
4,bogo,O,40s,40-60k,1


shape: (61042, 29)
offer_type
discount    30543
bogo        30499
Name: count, dtype: int64


In [3]:
# 기본 분포와 정상 전환율
def segment_rate_summary(df, group_col, target_col='is_normal_flow'):
    summary = (
        df.groupby(group_col)
        .agg(total=('person', 'count'), normal=('is_normal_flow', 'sum'))
        .reset_index()
    )
    summary['normal_rate(%)'] = (summary['normal'] / summary['total'] * 100).round(2)
    return summary.sort_values('normal_rate(%)', ascending=False)

print('전체 정상 전환율:', round(analysis_df['is_normal_flow'].mean() * 100, 2), '%')

print('\n성별 정상 전환율')
display(segment_rate_summary(analysis_df, 'gender'))

print('\n연령대 정상 전환율')
display(segment_rate_summary(analysis_df, 'age_group'))

print('\n소득수준 정상 전환율')
display(segment_rate_summary(analysis_df, 'income_group'))


전체 정상 전환율: 38.12 %

성별 정상 전환율


,gender,total,normal,normal_rate(%)
2,O,721,386,53.54
0,F,21918,10426,47.57
1,M,30562,11522,37.70
3,Unknown,7841,933,11.90



연령대 정상 전환율


,age_group,total,normal,normal_rate(%)
4,50s,12692,5629,44.35
5,60s,10780,4710,43.69
3,40s,8241,3600,43.68
2,30s,5515,2093,37.95
1,20s,4997,1557,31.16
6,70+,18095,5474,30.25
0,10s,722,204,28.25



소득수준 정상 전환율


,income_group,total,normal,normal_rate(%)
4,80-100k,9364,4900,52.33
1,100k+,3952,1828,46.26
3,60-80k,16712,7530,45.06
2,40-60k,16101,6000,37.26
0,0-40k,7072,2076,29.36
5,Unknown,7841,933,11.90


In [4]:
def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))


def adjusted_residuals(observed, expected):
    obs = observed.to_numpy(dtype=float)
    n = obs.sum()

    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)

    denom = np.sqrt(expected * (1 - row_sums / n) * (1 - col_sums / n))
    resid = np.divide(obs - expected, denom, out=np.zeros_like(obs, dtype=float), where=denom != 0)
    return pd.DataFrame(resid, index=observed.index, columns=observed.columns)


def run_chi2_test(df, group_col, target_col='is_normal_flow'):
    table = pd.crosstab(df[group_col], df[target_col])
    chi2, p_value, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    v = cramers_v(chi2, n, table.shape[0], table.shape[1])
    resid = adjusted_residuals(table, expected)

    print('=' * 80)
    print(f'{group_col} vs {target_col}')
    print('=' * 80)
    print('[교차표]')
    display(table)

    print('[카이제곱 검정]')
    print(f'chi2 = {chi2:.4f}')
    print(f'p-value = {p_value:.4e}')
    print(f'dof = {dof}')
    print(f"Cramer's V = {v:.4f}")

    print('[조정된 잔차]')
    display(resid.round(3))

    return {
        'group_col': group_col,
        'target_col': target_col,
        'chi2': chi2,
        'p_value': p_value,
        'dof': dof,
        'cramers_v': v,
    }


def pairwise_proportion_tests(df, group_col, target_col='is_normal_flow'):
    rows = []
    levels = sorted(df[group_col].dropna().astype(str).unique())

    for a, b in combinations(levels, 2):
        sub = df[df[group_col].astype(str).isin([a, b])].copy()
        count = np.array([
            sub[sub[group_col].astype(str) == a][target_col].sum(),
            sub[sub[group_col].astype(str) == b][target_col].sum(),
        ])
        nobs = np.array([
            (sub[group_col].astype(str) == a).sum(),
            (sub[group_col].astype(str) == b).sum(),
        ])
        stat, p_value = proportions_ztest(count, nobs)
        rows.append({
            'group_col': group_col,
            'level_a': a,
            'level_b': b,
            'z_stat': stat,
            'p_value': p_value,
        })

    result = pd.DataFrame(rows)
    if not result.empty:
        result['p_bonferroni'] = np.minimum(result['p_value'] * len(result), 1.0)
        result = result.sort_values('p_value').reset_index(drop=True)
    return result


In [5]:
# 전체 검정: 고객 특성과 정상 전환 여부의 관련성
overall_results = [
    run_chi2_test(analysis_df, 'gender'),
    run_chi2_test(analysis_df, 'age_group'),
    run_chi2_test(analysis_df, 'income_group'),
]

overall_summary = pd.DataFrame(overall_results).sort_values('cramers_v', ascending=False).reset_index(drop=True)
display(overall_summary)


gender vs is_normal_flow
[교차표]


is_normal_flow,0,1
gender,,
F,11492,10426
M,19040,11522
O,335,386
Unknown,6908,933


[카이제곱 검정]
chi2 = 3189.9316
p-value = 0.0000e+00
dof = 3
Cramer's V = 0.2286
[조정된 잔차]


is_normal_flow,0,1
gender,,
F,-35.989,35.989
M,2.119,-2.119
O,-8.576,8.576
Unknown,51.202,-51.202


age_group vs is_normal_flow
[교차표]


...

income_group vs is_normal_flow
[교차표]


is_normal_flow,0,1
income_group,,
0-40k,4996,2076
100k+,2124,1828
40-60k,10101,6000
60-80k,9182,7530
80-100k,4464,4900
Unknown,6908,933


[카이제곱 검정]
chi2 = 3774.0828
p-value = 0.0000e+00
dof = 5
Cramer's V = 0.2487
[조정된 잔차]


is_normal_flow,0,1
income_group,,
0-40k,16.133,-16.133
100k+,-10.893,10.893
40-60k,2.593,-2.593
60-80k,-21.680,21.680
80-100k,-30.775,30.775
Unknown,51.202,-51.202


,group_col,target_col,chi2,p_value,dof,cramers_v
0,income_group,is_normal_flow,3774.082778,0.000000e+00,5,0.248652
1,gender,is_normal_flow,3189.931618,0.000000e+00,3,0.228600
2,age_group,is_normal_flow,1066.426593,3.826560e-227,6,0.132176


In [6]:
# 세부 segment 검정: 각 고객 특성 내부에서 정상 전환 비율 차이가 있는지 확인
gender_pairwise = pairwise_proportion_tests(analysis_df, 'gender')
age_pairwise = pairwise_proportion_tests(analysis_df, 'age_group')
income_pairwise = pairwise_proportion_tests(analysis_df, 'income_group')

print('성별 pairwise 비교')
display(gender_pairwise)

print('연령대 pairwise 비교')
display(age_pairwise)

print('소득수준 pairwise 비교')
display(income_pairwise)


성별 pairwise 비교


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,gender,F,Unknown,55.796886,0.000000e+00,0.000000e+00
1,gender,M,Unknown,43.539053,0.000000e+00,0.000000e+00
2,gender,O,Unknown,29.637851,4.863897e-193,2.918338e-192
3,gender,F,M,22.601296,4.208387e-113,2.525032e-112
4,gender,M,O,-8.656201,4.877488e-18,2.926493e-17
5,gender,F,O,-3.157002,1.594000e-03,9.564000e-03


연령대 pairwise 비교


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,age_group,50s,70+,25.360007,6.969770e-142,1.463652e-140
1,age_group,60s,70+,23.120219,2.899143e-118,6.088200e-117
2,age_group,40s,70+,21.269548,2.173628e-100,4.564619e-99
3,age_group,20s,50s,-16.083636,3.322909e-58,6.978109e-57
4,age_group,20s,60s,-14.966579,1.214083e-50,2.549573e-49
5,age_group,20s,40s,-14.325628,1.513544e-46,3.178442e-45
6,age_group,30s,70+,10.726625,7.633344e-27,1.603002e-25
7,age_group,10s,50s,-8.486321,2.132812e-17,4.478905e-16
8,age_group,10s,60s,-8.117795,4.747279e-16,9.969285e-15
9,age_group,10s,40s,-8.043125,8.757567e-16,1.839089e-14


소득수준 pairwise 비교


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,income_group,60-80k,Unknown,50.969018,0.000000e+00,0.000000e+00
1,income_group,80-100k,Unknown,55.792154,0.000000e+00,0.000000e+00
2,income_group,40-60k,Unknown,40.610701,0.000000e+00,0.000000e+00
3,income_group,100k+,Unknown,41.589589,0.000000e+00,0.000000e+00
4,income_group,0-40k,80-100k,-29.503000,2.634821e-191,3.952231e-190
5,income_group,0-40k,Unknown,26.523559,5.185480e-155,7.778220e-154
6,income_group,40-60k,80-100k,-23.425052,2.374658e-121,3.561986e-120
7,income_group,0-40k,60-80k,-22.558521,1.107641e-112,1.661462e-111
8,income_group,0-40k,100k+,-17.792514,8.077260e-71,1.211589e-69
9,income_group,40-60k,60-80k,-14.335542,1.312181e-46,1.968272e-45


In [7]:
# 유의한 세부 segment만 따로 보기
def significant_pairs(df, alpha=0.05):
    if df.empty:
        return df
    return df[df['p_bonferroni'] < alpha].reset_index(drop=True)

print('성별 유의한 pairwise')
display(significant_pairs(gender_pairwise))

print('연령대 유의한 pairwise')
display(significant_pairs(age_pairwise))

print('소득수준 유의한 pairwise')
display(significant_pairs(income_pairwise))


성별 유의한 pairwise


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,gender,F,Unknown,55.796886,0.000000e+00,0.000000e+00
1,gender,M,Unknown,43.539053,0.000000e+00,0.000000e+00
2,gender,O,Unknown,29.637851,4.863897e-193,2.918338e-192
3,gender,F,M,22.601296,4.208387e-113,2.525032e-112
4,gender,M,O,-8.656201,4.877488e-18,2.926493e-17
5,gender,F,O,-3.157002,1.594000e-03,9.564000e-03


연령대 유의한 pairwise


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,age_group,50s,70+,25.360007,6.969770e-142,1.463652e-140
1,age_group,60s,70+,23.120219,2.899143e-118,6.088200e-117
2,age_group,40s,70+,21.269548,2.173628e-100,4.564619e-99
3,age_group,20s,50s,-16.083636,3.322909e-58,6.978109e-57
4,age_group,20s,60s,-14.966579,1.214083e-50,2.549573e-49
5,age_group,20s,40s,-14.325628,1.513544e-46,3.178442e-45
6,age_group,30s,70+,10.726625,7.633344e-27,1.603002e-25
7,age_group,10s,50s,-8.486321,2.132812e-17,4.478905e-16
8,age_group,10s,60s,-8.117795,4.747279e-16,9.969285e-15
9,age_group,10s,40s,-8.043125,8.757567e-16,1.839089e-14


소득수준 유의한 pairwise


,group_col,level_a,level_b,z_stat,p_value,p_bonferroni
0,income_group,60-80k,Unknown,50.969018,0.000000e+00,0.000000e+00
1,income_group,80-100k,Unknown,55.792154,0.000000e+00,0.000000e+00
2,income_group,40-60k,Unknown,40.610701,0.000000e+00,0.000000e+00
3,income_group,100k+,Unknown,41.589589,0.000000e+00,0.000000e+00
4,income_group,0-40k,80-100k,-29.503000,2.634821e-191,3.952231e-190
5,income_group,0-40k,Unknown,26.523559,5.185480e-155,7.778220e-154
6,income_group,40-60k,80-100k,-23.425052,2.374658e-121,3.561986e-120
7,income_group,0-40k,60-80k,-22.558521,1.107641e-112,1.661462e-111
8,income_group,0-40k,100k+,-17.792514,8.077260e-71,1.211589e-69
9,income_group,40-60k,60-80k,-14.335542,1.312181e-46,1.968272e-45


## 해석 포인트

- 전체 검정은 `gender`, `age_group`, `income_group`이 `is_normal_flow`와 관련이 있는지 확인한다.
- 세부 segment 검정은 각 고객 특성 내부에서 어떤 집단끼리 정상 전환 비율 차이가 나는지 보여준다.
- `p-value`가 작을수록 정상 전환 비율 차이가 뚜렷하고, `p_bonferroni`는 다중비교 보정 후에도 유의한지를 확인한다.
- 최종 발표에서는 `고객 특성은 정상 전환 여부와 유의한 관련이 있으며, 특히 소득수준과 연령대의 설명력이 더 크다`로 요약하면 된다.
